In [1]:
import logging
import claytonlib as clayton
from claytonlib.chart import (
    ChartSafariInput,
    chart_safari,
    evaluate_chart,
    evaluate_chart_top_10,
    chart_config,
    STRATEGY_ONLY_BALLS,
    STRATEGY_ONE_MUD,
    STRATEGY_SIX_BAIT,
    CRITERIA_CAPTURE,
    CRITERIA_WONT_FLEE_10_TURNS,
    SlidingWindowSum,
    NormalWindow,
)
from claytonlib.safari import safari_pokemon_by_name
from claytonlib.compass import (
    CompassSafariInput,
    compass_safari,
    compass_config,
)

In [13]:
# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG detail
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)

# --- Target ---
POKEMON_NAME = 'metang'
KEY_SEED     = 0xEC1504DC # delay = 1244
setup_delay_seconds = 300 # 5 minutes of setup
max_target_seconds = 3600

pokemon  = safari_pokemon_by_name(POKEMON_NAME)
strategy = STRATEGY_ONLY_BALLS
criteria = CRITERIA_CAPTURE

# --- Chart inputs ---
# setup_delay_seconds: real-world seconds between hitting the key seed and
# the first reachable encounter (seed confirmation, RNG advance, Sweet Scent, etc.)
# max_target_seconds: last second offset to evaluate (defines the chart window)
inputs = ChartSafariInput(
    key_seed             = KEY_SEED,
    setup_delay_seconds  = setup_delay_seconds,
    max_target_seconds   = max_target_seconds,
    strategy             = strategy,
    criteria             = criteria,
    pokemon              = pokemon,
)

# --- chart_config options ---
chart_config().evaluation_frames_per_write_cycle = 60
chart_config().resume_validation_enabled         = False
chart_config().evaluation_chains_per_write_cycle = 50
# --- Evaluation strategies ---
# sigma_frames=12 ≈ 0.4 s timing window (1 frame = 2 delay units)
# eval_strategy = NormalWindow(sigma_frames=12)
eval_strategy = SlidingWindowSum(window=13)  # uniform 2-second window

# --- Compass inputs ---
# Set COMPASS_TARGET_DELAY to a delay value and COMPASS_INITIAL_TIME to the
# matching "Initial Time" from the evaluate_chart output above.
import datetime as dt
COMPASS_TARGET_DELAY = 48944 # 48948                          # e.g. 19232
COMPASS_INITIAL_TIME = dt.datetime(2000, 5, 27, 21, 57, 44)                          # e.g. dt.datetime(2000, 1, 15, 12, 4, 42)
COMPASS_WINDOW       = 60                            # search ±60 delay units (~1 second)

if COMPASS_TARGET_DELAY is not None and COMPASS_INITIAL_TIME is not None:
    compass_inputs = CompassSafariInput.from_chart(
        inputs,
        window        = COMPASS_WINDOW,
        initial_time  = COMPASS_INITIAL_TIME,
        target_delay  = COMPASS_TARGET_DELAY,
        # target_seed = 0x...  # optional: verified against target_delay + initial_time
        evaluation_strategy = STRATEGY_ONLY_BALLS,  # uncomment to show success column
    )

In [6]:
chart_safari(inputs)

10:09:54 INFO claytonlib.chart: All chains already complete, nothing to do.


In [3]:
evaluate_chart_top_10(inputs, eval_strategy)

13:35:00 INFO claytonlib.chart: top10 across 2858 chain(s): 0.149s
13:35:00 INFO claytonlib.chart: evaluate_chart complete in 1.258s


Top 10 (by score)
 #        Score(p)    Delay      Time        Δ Time  Initial Time
-----------------------------------------------------------------
 1      4.4(33.8%)    48948  22:10:59   13m 15.067s  2000-05-27 21:57:44
 2     4.37(33.6%)   216928  22:57:58   59m 54.733s  2000-06-29 21:58:04
 3     4.37(33.6%)   216930  22:57:58   59m 54.767s  2000-06-29 21:58:04
 4     4.37(33.6%)   216932  22:57:58   59m 54.800s  2000-06-29 21:58:04
 5     4.07(31.3%)   206110  21:57:50   56m 54.433s  2000-06-30 21:00:56
 6     4.07(31.3%)   206112  21:57:50   56m 54.467s  2000-06-30 21:00:56
 7     4.07(31.3%)   206114  21:57:50   56m 54.500s  2000-06-30 21:00:56
 8      4.0(30.8%)   119516  21:58:58   32m 51.200s  2000-07-29 21:26:07
 9      4.0(30.8%)   119518  21:58:58   32m 51.233s  2000-07-29 21:26:07
10      4.0(30.8%)   119520  21:58:58   32m 51.267s  2000-07-29 21:26:07

Best 10 (highest score at each successively lower delay)
 #        Score(p)    Delay      Time        Δ Time  Initial T

In [ ]:
print(input("Input"))

In [14]:
compass_safari(compass_inputs)

=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit         Metang is angry!
  M / a  Mud, crit (Anger)    Metang is beside itself with anger!
  b      Bait, no crit        Metang is eating!
  B / e  Bait, crit (Eating)  Metang is busy eating!
  0      Ball, 0 shakes       Oh, no! The Pokémon broke free!
  1      Ball, 1 shake        Aww! It appeared to be caught!
  2      Ball, 2 shakes       Aargh! Almost had it!
  3      Ball, 3 shakes       Shoot! It was so close, too!
  C      Captured (ends)      Gotcha! Metang was caught!
  F      Fled (ends)          Metang fled!
  u      Undo last action     —
  ?x     Uncertain result     —
  Spaces and commas in input are ignored.


Seeds: 119 / 119 remaining
Path:  (none)
Balls: 30
   #        Seed    Delay      Δ
   1. 0xCC16BF30    48944      0  ← target
   2. 0x9216BF32    48946     +2
   3. 0xCB16BF2E    48942     -2
   4. 0xCC16BF2E    48942     -2
   5. 0xCC16BF32    48946     +2



>>  0



Seeds: 83 / 119 remaining
Path:  0
Balls: 29
   #        Seed    Delay      Δ
   1. 0xCC16BF30    48944      0  ← target
   2. 0xCB16BF2E    48942     -2
   3. 0x9216BF34    48948     +4
   4. 0xCB16BF2C    48940     -4
   5. 0xCC16BF2C    48940     -4



>>  1



Seeds: 13 / 119 remaining
Path:  01
Balls: 28
   #        Seed    Delay      Δ
   1. 0xCC16BF30    48944      0  ← target
   2. 0xCB16BF2A    48938     -6
   3. 0xCC16BF38    48952     +8
   4. 0x9216BF3A    48954    +10
   5. 0xCC16BF26    48934    -10


...


Seeds: 6 / 119 remaining
Path:  010
Balls: 27
   #        Seed    Delay      Δ  Success
   1. 0xCC16BF30    48944      0  yes  ← target
   2. 0xCB16BF2A    48938     -6  no
   3. 0x9216BF42    48962    +18  no
   4. 0xCC16BF58    48984    +40  no
   5. 0xCB16BF00    48896    -48  no



>>  0



Seeds: 3 / 119 remaining
Path:  0100
Balls: 26
   #        Seed    Delay      Δ  Success
   1. 0xCC16BF30    48944      0  yes  ← target
   2. 0x9216BF42    48962    +18  no
   3. 0xCB16BF00    48896    -48  no



>>  1



Seeds: 0 / 119 remaining
Path:  01001
Balls: 30
   #        Seed    Delay      Δ  Success

No matching seed found in window ±60.
Consider expanding the search window or checking for input errors.
